# GOLD (Batch)

## G1) sales_daily_summary — revenue by date and category

In [0]:
from pyspark.sql import functions as F

catalog = "abs_dev"
db_name = "abs_db"


# ============================================================
# 1. CREATE DAILY SALES SUMMARY TABLE
# ============================================================

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.{db_name}.sales_daily_summary (
    order_date       DATE,
    product_category STRING,
    num_orders       BIGINT,
    num_line_items   BIGINT,
    units_sold       BIGINT,
    gross_revenue    DECIMAL(18,2),
    total_discount   DECIMAL(18,2),
    net_revenue      DECIMAL(18,2)
)
USING DELTA
""")


# ============================================================
# 2. BUILD DAILY SALES SUMMARY
# ============================================================

sales_daily_summary = (
    spark.read.table(f"{catalog}.{db_name}.sales_enriched")

        .groupBy(
            "order_date",
            "product_category"
        )

        .agg(
            F.countDistinct("order_id")
                .cast("bigint")
                .alias("num_orders"),

            F.count(F.lit(1))
                .cast("bigint")
                .alias("num_line_items"),

            F.sum("quantity")
                .cast("bigint")
                .alias("units_sold"),

            F.sum("total_amount")
                .cast("decimal(18,2)")
                .alias("gross_revenue"),

            F.sum("discount_amount")
                .cast("decimal(18,2)")
                .alias("total_discount"),

            F.sum("net_amount")
                .cast("decimal(18,2)")
                .alias("net_revenue")
        )
)


# ============================================================
# 3. CREATE TEMPORARY SOURCE VIEW
# ============================================================

sales_daily_summary.createOrReplaceTempView(
    "sales_daily_summary_source"
)


# ============================================================
# 4. RESTATABLE MERGE
# ============================================================
# Recomputes the daily/category totals from sales_enriched.
# This allows existing totals to be corrected when late-arriving
# sales are loaded into the underlying sales table.

spark.sql(f"""
MERGE INTO {catalog}.{db_name}.sales_daily_summary AS target
USING sales_daily_summary_source AS source

ON target.order_date = source.order_date
AND target.product_category <=> source.product_category

WHEN MATCHED THEN
UPDATE SET
    target.num_orders      = source.num_orders,
    target.num_line_items  = source.num_line_items,
    target.units_sold      = source.units_sold,
    target.gross_revenue   = source.gross_revenue,
    target.total_discount  = source.total_discount,
    target.net_revenue     = source.net_revenue

WHEN NOT MATCHED THEN
INSERT (
    order_date,
    product_category,
    num_orders,
    num_line_items,
    units_sold,
    gross_revenue,
    total_discount,
    net_revenue
)
VALUES (
    source.order_date,
    source.product_category,
    source.num_orders,
    source.num_line_items,
    source.units_sold,
    source.gross_revenue,
    source.total_discount,
    source.net_revenue
)
""")


# ============================================================
# 5. DISPLAY RESULT
# ============================================================

display(
    spark.table(f"{catalog}.{db_name}.sales_daily_summary")
         .orderBy("order_date", "product_category")
         .limit(20)
)

order_date,product_category,num_orders,num_line_items,units_sold,gross_revenue,total_discount,net_revenue
2023-12-01,null,1,1,5,434.55,7.70,426.85
2023-12-01,Electronics,7,7,22,3667.92,67.00,3600.92
2023-12-01,Home Goods,5,5,15,5088.57,50.67,5037.90
2023-12-02,null,2,2,8,698.90,8.70,690.20
2023-12-02,Electronics,4,4,10,1529.66,45.74,1483.92
2023-12-02,Home Goods,6,6,16,3887.96,46.79,3841.17
2023-12-03,null,2,2,6,513.06,17.64,495.42
2023-12-03,Electronics,1,1,2,464.51,7.51,457.00
2023-12-03,Home Goods,2,2,4,1258.30,18.40,1239.90
2023-12-04,null,2,2,6,507.29,23.41,483.88


## G2) customer_sales_summary — lifetime value per customer

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.{db_name}.customer_sales_summary (
  customer_id       STRING,
  customer_name     STRING,
  customer_city     STRING,
  customer_state    STRING,
  first_order_date  DATE,
  last_order_date   DATE,
  num_orders        BIGINT,
  units_purchased   BIGINT,
  total_discount    DECIMAL(18,2),
  net_revenue       DECIMAL(18,2),
  avg_order_value   DECIMAL(18,2)
) USING DELTA
""")

customer_sales_summary = (
    spark.read.table(f"{catalog}.{db_name}.sales_enriched")
        .filter(F.col("customer_id").isNotNull())
        .groupBy("customer_id", "customer_name", "customer_city", "customer_state")
        .agg(
            F.min("order_date").alias("first_order_date"),
            F.max("order_date").alias("last_order_date"),
            F.countDistinct("order_id").alias("num_orders"),
            F.sum("quantity").cast("bigint").alias("units_purchased"),
            F.sum("discount_amount").cast("decimal(18,2)").alias("total_discount"),
            F.sum("net_amount").cast("decimal(18,2)").alias("net_revenue"),
        )
        .withColumn(
            "avg_order_value",
            (F.col("net_revenue") / F.col("num_orders")).cast("decimal(18,2)")
        )
)

customer_sales_summary.createOrReplaceTempView("customer_sales_summary_source")

spark.sql(f"""
MERGE INTO {catalog}.{db_name}.customer_sales_summary a
USING customer_sales_summary_source b
ON a.customer_id = b.customer_id
WHEN MATCHED     THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
""")

display(spark.read.table(f"{catalog}.{db_name}.customer_sales_summary").orderBy(F.col("net_revenue").desc()).limit(20))

customer_id,customer_name,customer_city,customer_state,first_order_date,last_order_date,num_orders,units_purchased,total_discount,net_revenue,avg_order_value
1,Paul Shelton,Smithborough,AK,2023-12-01,2023-12-05,12,38,119.32,8951.67,745.97
2,Brandon Brooks,New York,NY,2023-12-01,2023-12-05,12,40,106.54,7920.22,660.02
5,Beth Turner,Priceton,CO,2023-12-01,2023-12-05,11,28,86.02,5820.80,529.16
4,Derek Hawkins,Martinezshire,ND,2023-12-01,2023-12-05,8,24,46.61,4887.31,610.91
3,null,null,null,2023-12-01,2023-12-05,7,21,96.22,3594.65,513.52


## G3) product_performance (view)

In [0]:
spark.sql(f"""CREATE OR REPLACE VIEW {catalog}.{db_name}.product_performance AS
                SELECT p.product_id,
                       p.product_name,
                       p.product_category,
                       p.product_price,
                       COUNT(DISTINCT s.order_id)        AS num_orders,
                       COALESCE(SUM(s.quantity), 0)      AS units_sold,
                       COALESCE(SUM(s.net_amount), 0)    AS net_revenue,
                       COALESCE(SUM(s.discount_amount), 0) AS total_discount,
                       ROUND(COALESCE(SUM(s.discount_amount), 0)
                             / NULLIF(SUM(s.total_amount), 0) * 100, 2) AS discount_rate_pct,
                       MAX(s.order_date)                 AS last_sold_date
                FROM {catalog}.{db_name}.products p
                LEFT JOIN {catalog}.{db_name}.sales s
                  ON p.product_id = s.product_id
                GROUP BY p.product_id, p.product_name, p.product_category, p.product_price
                ORDER BY net_revenue DESC
""")

display(spark.read.table(f"{catalog}.{db_name}.product_performance").limit(20))

product_id,product_name,product_category,product_price,num_orders,units_sold,net_revenue,total_discount,discount_rate_pct,last_sold_date
3,where,Home Goods,555.38,9,30,12909.14,91.13,0.70,2023-12-05
5,decide,Home Goods,201.97,13,31,6067.11,96.98,1.57,2023-12-05
4,economic,Electronics,123.91,14,43,5069.15,129.49,2.49,2023-12-05
2,rise,Electronics,236.01,7,22,5032.90,79.66,1.56,2023-12-05


## Validation — row counts and referential checks
Run after a full pass. Re-run the whole notebook and these numbers should not move.

In [0]:
for t in ["sales_bz", "products_bz", "customers_bz",
          "products", "customers", "sales", "sales_enriched",
          "sales_daily_summary", "customer_sales_summary"]:
    print(f"{t:>25}: {spark.table(f'{catalog}.{db_name}.{t}').count():,}")

orphan_products = spark.sql(f"""
    SELECT COUNT(*) FROM {catalog}.{db_name}.sales_enriched WHERE product_name IS NULL
""").collect()[0][0]

orphan_customers = spark.sql(f"""
    SELECT COUNT(*) FROM {catalog}.{db_name}.sales_enriched WHERE customer_name IS NULL
""").collect()[0][0]

dupes = spark.sql(f"""
    SELECT COUNT(*) FROM (
      SELECT order_id, product_id, transaction_id
      FROM {catalog}.{db_name}.sales
      GROUP BY order_id, product_id, transaction_id
      HAVING COUNT(*) > 1
    )
""").collect()[0][0]

print(f"\nsales lines with no matching product : {orphan_products:,}")
print(f"sales lines with no matching customer: {orphan_customers:,}")
print(f"duplicate sales line grains          : {dupes:,}  (must be 0)")

                 sales_bz: 140
              products_bz: 14
             customers_bz: 8
                 products: 4
                customers: 5
                    sales: 50
           sales_enriched: 50
      sales_daily_summary: 14
   customer_sales_summary: 5

sales lines with no matching product : 7
sales lines with no matching customer: 7
duplicate sales line grains          : 0  (must be 0)
